<a href="https://colab.research.google.com/github/Dambril/Dambril/blob/main/pracFinalDL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Librerias**

In [ ]:
!pip install tensorflow librosa soundfile opencv-python-headless

from google.colab import drive
import cv2
import os
import librosa
import numpy as np
import soundfile as sf
import shutil
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, LSTM, GRU, Bidirectional, Concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

drive.mount('/content/drive')


Mounted at /content/drive


# **Funcion de extracción y preprocesamiento del datasets**

In [ ]:
#CONFIG
rawDatasetPath = '/content/drive/MyDrive/dl_2025/CREMA-D/VideoFlash'
outputBasePath = '/content/drive/MyDrive/dl_2025/datos_preprocesados'

# Mapeo para las carpetas
mapeoEmo = {
    'ANG': 'enojado',
    'HAP': 'feliz',
    'NEU': 'neutral',
    'SAD': 'triste',
    'FEA': 'asustado',
    'DIS': 'disgustado'
}

def procesarDataset():
    # Crear carpetas
    for emocion in mapeoEmo.values():
        os.makedirs(os.path.join(outputBasePath, 'visual', emocion), exist_ok=True)
        os.makedirs(os.path.join(outputBasePath, 'audio', emocion), exist_ok=True)

    if not os.path.exists(rawDatasetPath):
        print(f"Error: La ruta del dataset '{rawDatasetPath}' no existe.")
        return

    files = os.listdir(rawDatasetPath)

    for filename in files:
        if not filename.endswith('.flv'): continue

        #Identificar emoción por el nombre del archivo (como 1001_DFA_ANG_XX.flv)
        code = filename.split('_')[2]
        if code not in mapeoEmo: continue

        folderEmo = mapeoEmo[code]
        inputFile = os.path.join(rawDatasetPath, filename)
        baseName = filename[:-4]

        #1. PROCESAR AUDIO
        try:
            #Extraer audio con librosa
            y, sr = librosa.load(inputFile, sr=16000) # 16kHz
            audio_out_path = os.path.join(outputBasePath, 'audio', folderEmo, f"{baseName}.wav")
            sf.write(audio_out_path, y, sr)
        except Exception as e:
            print(f"Error audio en {filename}: {e}")

        #2. PROCESAR VIDEO
        try:
            cap = cv2.VideoCapture(inputFile)
            cascadeCara = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

            frameCont = 0
            savedCont = 0

            while True:
                ret, frame = cap.read()
                if not ret: break

                # Guardar solo 1 de cada 10 frames para ahorrar espacio
                if frameCont % 10 == 0:
                    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                    faces = cascadeCara.detectMultiScale(gray, 1.1, 4)

                    for (x, y, w, h) in faces:
                        # Recortar rostro
                        face = gray[y:y+h, x:x+w]
                        # Redimensionar a 48x48 (Estándar CNN ligera)
                        redimension = cv2.resize(face, (48, 48))

                        img_name = f"{baseName}_frame{savedCont}.jpg"
                        img_path = os.path.join(outputBasePath, 'visual', folderEmo, img_name)
                        cv2.imwrite(img_path, redimension)
                        savedCont += 1
                        break # Solo 1 cara por frame

                frameCont += 1
            cap.release()
        except Exception as e:
            print(f"Error video en {filename}: {e}")

    print("¡Procesamiento completado!")

# Ejecutar
procesarDataset()

Se truncaron las últimas líneas 5000 del resultado de transmisión.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipython-input-3124642725.py:41: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(inputFile, sr=16000) # 16kHz
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipython-input-3124642725.py:41: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(inputFile, sr=16000) # 16kHz
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duratio

# **Modelos individuales**

**Visual**

In [ ]:
import shutil
import os

# Rutas
origen_drive = '/content/drive/MyDrive/dl_2025/datos_preprocesados/visual'
destino_local = '/content/datosModLocales'

print("Copiando datos de Drive al entorno local...")

# Copiar carpeta entera
if os.path.exists(destino_local):
    shutil.rmtree(destino_local) # Limpiar si ya existe para evitar mezcla de datos
shutil.copytree(origen_drive, destino_local)

print("¡Copia terminada! Ahora entrena usando 'destino_local'")

Copiando datos de Drive al entorno local...
¡Copia terminada! Ahora entrena usando 'destino_local'


In [ ]:
# Generadores de datos (Cargan las imágenes directo de tus carpetas)
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
baseDirec = '/content/datosModLocales'

genTrain = datagen.flow_from_directory(
    baseDirec, target_size=(48, 48), color_mode='grayscale',
    batch_size=64, class_mode='categorical', subset='training')

genVal = datagen.flow_from_directory(
    baseDirec, target_size=(48, 48), color_mode='grayscale',
    batch_size=64, class_mode='categorical', subset='validation')

# Modelo CNN Simple
modeloVisual = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(48,48,1)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(6, activation='softmax') # 6 emociones
])

modeloVisual.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
modeloVisual.fit(genTrain, epochs=40, validation_data=genVal)
modeloVisual.save('modelo_visual_final.h5')

Found 11662 images belonging to 6 classes.
Found 2913 images belonging to 6 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/40
183/183 ━━━━━━━━━━━━━━━━━━━━ 48s 235ms/step - accuracy: 0.1946 - loss: 1.7820 - val_accuracy: 0.2650 - val_loss: 1.7050
Epoch 2/40
183/183 ━━━━━━━━━━━━━━━━━━━━ 38s 210ms/step - accuracy: 0.3840 - loss: 1.5092 - val_accuracy: 0.3169 - val_loss: 1.6703
Epoch 3/40
183/183 ━━━━━━━━━━━━━━━━━━━━ 39s 212ms/step - accuracy: 0.4607 - loss: 1.3321 - val_accuracy: 0.3375 - val_loss: 1.6820
Epoch 4/40
183/183 ━━━━━━━━━━━━━━━━━━━━ 40s 216ms/step - accuracy: 0.5146 - loss: 1.2349 - val_accuracy: 0.3361 - val_loss: 1.8370
Epoch 5/40
183/183 ━━━━━━━━━━━━━━━━━━━━ 39s 213ms/step - accuracy: 0.5405 - loss: 1.1868 - val_accuracy: 0.2966 - val_loss: 1.9136
Epoch 6/40
183/183 ━━━━━━━━━━━━━━━━━━━━ 38s 210ms/step - accuracy: 0.5546 - loss: 1.1230 - val_accuracy: 0.3371 - val_loss: 2.0064
Epoch 7/40
183/183 ━━━━━━━━━━━━━━━━━━━━ 41s 225ms/step - accuracy: 0.5911 - loss: 1.0664 - val_accuracy: 0.3419 - val_loss: 1.9024
Epoch 8/40
183/183 ━━━━━━━━━━━━━━━━━━━━ 38s 208ms/step - accuracy: 0.5982 - loss: 1

**Auditivo**

In [ ]:
import shutil
import os

# --- Configuración de Rutas de Audio ---
origen_drive_audio = '/content/drive/MyDrive/dl_2025/datos_preprocesados/audio'
destino_local_audio = '/content/datosModAuLocales'

# Limpiar carpeta si ya existe
if os.path.exists(destino_local_audio):
    print(f"Limpiando carpeta existente: {destino_local_audio}")
    shutil.rmtree(destino_local_audio)

print("Copiando datos de audio de Drive a la memoria local de la GPU...")

# Copiar todo el árbol de directorios de Drive al disco local de la VM
shutil.copytree(origen_drive_audio, destino_local_audio)

print("¡Copia de Audio terminada!")

Copiando datos de audio de Drive a la memoria local de la GPU...
¡Copia de Audio terminada!


In [ ]:
# --- 1. CONFIGURACIÓN ---
baseDirec = '/content/datosModAuLocales'

# Parámetros de Audio
DURACION = 3 # Segundos (estándar para CREMA-D)
SAMPLERATE = 16000 # Hz
N_MFCC = 40 # Número de características a extraer
SAMPLES_ESPERADOS = SAMPLERATE * DURACION # 16000 * 3 = 48000 muestras

# Mapeo de emociones (debe coincidir con tus carpetas)
emociones = ['enojado', 'asustado', 'feliz', 'neutral', 'triste', 'disgustado']

# --- 2. FUNCIÓN DE EXTRACCIÓN DE CARACTERÍSTICAS ---
def extraerCarac(file_path): #recuerda que cambiaste el nombre a spanish weee
    try:
        # Cargar audio
        y, sr = librosa.load(file_path, sr=SAMPLERATE, duration=DURACION)

        # Estandarizar longitud (Padding o Truncating)
        if len(y) < SAMPLES_ESPERADOS:
            # Rellenar con ceros si es muy corto
            y = np.pad(y, (0, SAMPLES_ESPERADOS - len(y)), mode='constant')
        else:
            # Recortar si es muy largo
            y = y[:SAMPLES_ESPERADOS]

        # Esto convierte el audio en una "imagen" de calor del sonido
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)

        # Transponer para tener (Time_Steps, Features) que es lo que pide la LSTM
        # Resultado shape: (Tiempo, 40)
        return mfcc.T
    except Exception as e:
        print(f"Error procesando {file_path}: {e}")
        return None

# --- 3. CARGA DE DATOS ---
print("Iniciando carga de audios... esto puede tardar unos minutos.")
X = []
y = []

for emotion in emociones:
    emotion_path = os.path.join(baseDirec, emotion)
    if not os.path.exists(emotion_path):
        print(f"Advertencia: No se encontró la carpeta: {emotion_path}")
        continue

    print(f"Procesando emoción: {emotion}...")
    files = os.listdir(emotion_path)

    for file in files:
        if file.endswith('.wav'):
            file_path = os.path.join(emotion_path, file)
            features = extraerCarac(file_path)

            if features is not None:
                X.append(features)
                y.append(emotion)

# Convertir a Arrays de Numpy
X = np.array(X)
y = np.array(y)

print(f"Datos cargados. Shape de X: {X.shape}")
# Debería ser algo como (Numero_Audios, ~94, 40)
# ~94 viene de (48000 muestras / 512 hop_length)

# --- 4. PREPARACIÓN DE ETIQUETAS ---
le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_categorical = to_categorical(y_encoded) # One-hot encoding

# Guardar el orden de las clases para usarlo luego en la app web
print("Orden de clases detectado:", le.classes_)
np.save('classes.npy', le.classes_) # Guarda este archivo, lo necesitarás en local

# División Train/Test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y_categorical, test_size=0.2, random_state=42)

# --- 5. CONSTRUCCIÓN DEL MODELO RNN (LSTM) ---
model = Sequential([
    # Capa LSTM. input_shape=(Time_Steps, Features)
    LSTM(128, return_sequences=True, input_shape=(X.shape[1], N_MFCC)),
    BatchNormalization(),
    Dropout(0.3),

    LSTM(64), # Segunda capa LSTM
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(len(emociones), activation='softmax') # Salida: 6 emociones
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

# --- 6. ENTRENAMIENTO ---
print("Iniciando entrenamiento...")
history = model.fit(X_train, y_train,
                    validation_data=(X_test, y_test),
                    epochs=60,
                    batch_size=32)

# --- 7. GUARDAR MODELO ---
model.save('modelo_audio_final.h5')
print("Modelo guardado como 'modelo_audio_final.h5'")

Iniciando carga de audios... esto puede tardar unos minutos.
Procesando emoción: enojado...
Procesando emoción: asustado...
Procesando emoción: feliz...
Procesando emoción: neutral...
Procesando emoción: triste...
Procesando emoción: disgustado...
Datos cargados. Shape de X: (1800, 94, 40)
Orden de clases detectado: ['asustado' 'disgustado' 'enojado' 'feliz' 'neutral' 'triste']


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 94, 128)        │        86,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 94, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 94, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 141,254 (551.77 KB)

 Trainable params: 140,870 (550.27 KB)

 Non-trainable params: 384 (1.50 KB)

Iniciando entrenamiento...
Epoch 1/60
45/45 ━━━━━━━━━━━━━━━━━━━━ 14s 213ms/step - accuracy: 0.2286 - loss: 2.0528 - val_accuracy: 0.2167 - val_loss: 1.7606
Epoch 2/60
45/45 ━━━━━━━━━━━━━━━━━━━━ 9s 199ms/step - accuracy: 0.3104 - loss: 1.7666 - val_accuracy: 0.2417 - val_loss: 1.7435
Epoch 3/60
45/45 ━━━━━━━━━━━━━━━━━━━━ 11s 228ms/step - accuracy: 0.3551 - loss: 1.6362 - val_accuracy: 0.2639 - val_loss: 1.7205
Epoch 4/60
45/45 ━━━━━━━━━━━━━━━━━━━━ 9s 205ms/step - accuracy: 0.3761 - loss: 1.5720 - val_accuracy: 0.2917 - val_loss: 1.7041
Epoch 5/60
45/45 ━━━━━━━━━━━━━━━━━━━━ 9s 183ms/step - accuracy: 0.3612 - loss: 1.5704 - val_accuracy: 0.3333 - val_loss: 1.5957
Epoch 6/60
45/45 ━━━━━━━━━━━━━━━━━━━━ 10s 226ms/step - accuracy: 0.3877 - loss: 1.5454 - val_accuracy: 0.3528 - val_loss: 1.5639
Epoch 7/60
45/45 ━━━━━━━━━━━━━━━━━━━━ 9s 202ms/step - accuracy: 0.3647 - loss: 1.5150 - val_accuracy: 0.4250 - val_loss: 1.4729
Epoch 8/60
45/45 ━━━━━━━━━━━━━━━━━━━━ 10s 215ms/step - accuracy: 0.3941 - 

Modelo guardado como 'modelo_audio_final.h5'


In [ ]:
from google.colab import files

# Descargar el modelo visual
files.download('modelo_visual_final.h5')

# Descargar el modelo auditivo
files.download('modelo_audio_final.h5')

# Descargar las clases (el orden de las emociones)
files.download('classes.npy')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Fusión de los modelos**

In [ ]:
from sklearn.preprocessing import LabelEncoder
from google.colab import files
from io import BytesIO

print("Sube los archivos del modelo")
uploaded = files.upload() # Esto abre la ventana para subir archivos

# Parámetros (coincidir entrenamiento)
N_MFCC = 40
DURATION = 3
SAMPLE_RATE = 16000
IMG_SIZE = (48, 48)

#cargar Modelos
model_vis = tf.keras.models.load_model('modelo_visual_final.h5')
model_aud = tf.keras.models.load_model('modelo_audio_final.h5')

#cargar el orden de las clases (emociones)
EMOCIONES = np.load('classes.npy', allow_pickle=True)
print("Modelos y Clases cargadas. Orden de las clases:", EMOCIONES)

#cargar clasificador de rostros
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

Sube los archivos 'modelo_visual_final.h5', 'modelo_audio_final.h5' y 'classes.npy'


Saving classes.npy to classes.npy
Saving modelo_audio_final.h5 to modelo_audio_final.h5
Saving modelo_visual_final.h5 to modelo_visual_final.h5


Modelos y Clases cargadas. Orden de las clases: ['asustado' 'disgustado' 'enojado' 'feliz' 'neutral' 'triste']


In [ ]:
def predecir_emocion_multimodal(video_path):

    #PREDICCIÓN VISUAL (CNN) ---
    cap = cv2.VideoCapture(video_path)
    visual_preds_list = []

    # Analizar 1 frame por segundo (FPS)
    frame_rate = int(cap.get(cv2.CAP_PROP_FPS))
    count = 0

    while True:
        ret, frame = cap.read()
        if not ret: break

        # Procesar solo 1 frame cada 'frame_rate'
        if count % frame_rate == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, 1.1, 4)

            if len(faces) > 0:
                (x, y, w, h) = faces[0]
                face = gray[y:y+h, x:x+w]
                face_resized = cv2.resize(face, IMG_SIZE)

                # Normalizar y preparar para la CNN (Shape: (1, 48, 48, 1))
                normalized = face_resized / 255.0
                reshaped = np.reshape(normalized, (1, 48, 48, 1))

                # Predecir
                pred = model_vis.predict(reshaped, verbose=0)
                visual_preds_list.append(pred[0])
        count += 1

    cap.release()

    # Agregación: Obtener el promedio de las predicciones de todos los frames
    if not visual_preds_list:
        print("No se detectaron rostros en el video.")
        avg_visual = np.zeros(len(EMOCIONES))
    else:
        avg_visual = np.mean(visual_preds_list, axis=0) # Vector de 6 probabilidades

    #PREDICCIÓN AUDITIVA (RNN) ---
    try:
        y, sr = librosa.load(video_path, sr=SAMPLE_RATE, duration=DURATION)

        # Estandarización de longitud (igual que en el entrenamiento)
        expected_samples = SAMPLE_RATE * DURATION
        if len(y) < expected_samples:
            y = np.pad(y, (0, expected_samples - len(y)), mode='constant')
        else:
            y = y[:expected_samples]

        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
        mfcc_transposed = mfcc.T # (Time_Steps, Features)

        # Preparar para la RNN (Shape: (1, Time_Steps, Features))
        rnn_input = np.expand_dims(mfcc_transposed, axis=0)

        audio_pred = model_aud.predict(rnn_input, verbose=0)[0] # Vector de 6 probabilidades

    except Exception as e:
        print(f"Error al procesar audio: {e}")
        audio_pred = np.zeros(len(EMOCIONES))

    #FUSIÓN TARDÍA (Late Fusion) ---

    # 1. Suma de las probabilidades
    combined_predictions = avg_visual + audio_pred

    # 2. Normalizar o Promediar para obtener el vector final
    final_pred_vector = combined_predictions / 2

    # 3. Determinar la emoción final (el índice con la probabilidad más alta)
    idx_max = np.argmax(final_pred_vector)
    emocion_final = EMOCIONES[idx_max]

    # Devolver el resultado y el vector de predicciones para debug
    return emocion_final, final_pred_vector, avg_visual, audio_pred

In [ ]:
#Subr video local
print("Sube un video de prueba (ej. '.mp4' o '.avi')")
video_uploaded = files.upload()

# Asumimos que el primer archivo subido es el video
video_filename = list(video_uploaded.keys())[0]

# Ejecutar la predicción
emocion, fusion_vector, visual_vector, audio_vector = predecir_emocion_multimodal(video_filename)

# --- Impresión de Resultados ---

print("\n================ RESULTADOS DE INFERENCIA MULTIMODAL ================")
print(f"Video analizado: {video_filename}")
print("---------------------------------------------------------------------")

# Formatear el vector de predicción para mostrar porcentajes
def format_vector(vector):
    return {EMOCIONES[i]: f"{p*100:.2f}%" for i, p in enumerate(vector)}

print(f"Predicción FINAL (Fusión): {emocion.upper()}")
print(f"Vector de Fusión: {format_vector(fusion_vector)}")

print("\n--- Desglose por Modalidad ---")
print(f"Predicción VISUAL (CNN): {EMOCIONES[np.argmax(visual_vector)].upper()}")
print(f"Vector Visual: {format_vector(visual_vector)}")

print(f"Predicción AUDITIVA (RNN): {EMOCIONES[np.argmax(audio_vector)].upper()}")
print(f"Vector Auditivo: {format_vector(audio_vector)}")

print("\n=====================================================================")

Sube un video de prueba (ej. '.mp4' o '.avi')


Saving disgusto.mp4 to disgusto.mp4


/tmp/ipython-input-718355955.py:45: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(video_path, sr=SAMPLE_RATE, duration=DURATION)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)



================ RESULTADOS DE INFERENCIA MULTIMODAL ================
Video analizado: disgusto.mp4
---------------------------------------------------------------------
Predicción FINAL (Fusión): DISGUSTADO
Vector de Fusión: {np.str_('asustado'): '3.55%', np.str_('disgustado'): '45.85%', np.str_('enojado'): '38.88%', np.str_('feliz'): '11.45%', np.str_('neutral'): '0.24%', np.str_('triste'): '0.03%'}

--- Desglose por Modalidad ---
Predicción VISUAL (CNN): ENOJADO
Vector Visual: {np.str_('asustado'): '0.00%', np.str_('disgustado'): '0.00%', np.str_('enojado'): '77.24%', np.str_('feliz'): '22.55%', np.str_('neutral'): '0.21%', np.str_('triste'): '0.00%'}
Predicción AUDITIVA (RNN): DISGUSTADO
Vector Auditivo: {np.str_('asustado'): '7.10%', np.str_('disgustado'): '91.69%', np.str_('enojado'): '0.52%', np.str_('feliz'): '0.34%', np.str_('neutral'): '0.28%', np.str_('triste'): '0.07%'}

